# VCC 2026 · 4 — gene–gene attention within a perturbation

A transformer over the genes of each knockdown, with gene embeddings from PIE, trained to transfer knockdown effects across cell lines. This notebook starts from the raw cells: every source is rebuilt from single-cell counts with one cleaning rule and one labelling scheme (section 1); the model sections follow.

## 0. Session
Paths and the `run` / `bg` / `status` helpers.

In [ ]:
import os, sys, subprocess, shlex, json
from pathlib import Path
import pandas as pd
V = Path.home() / 'vcc'
ENV = dict(DATA=str(V / 'data'), LIB=str(V / 'lib'), LOGS=str(V / 'logs' / 'vcc_2026_4'), PY=sys.executable,
           WORK=str(V / 'lib' / 'vcc_2026_4'), CELLS=str(V / 'lib' / 'cells'), SCRIPTS=str(V / 'scripts'))
os.environ.update(ENV)
for d in ('LOGS', 'WORK'):
    os.makedirs(ENV[d], exist_ok=True)

def run(cmd):
    """Short step: run now, stream the output here, stop the cell on error."""
    p = subprocess.Popen(['bash', '-c', 'set -eo pipefail; cd ~/vcc\n' + cmd], env=os.environ,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'step failed (exit {p.returncode}) - see the output above')

def bg(name, cmd):
    """Long step: run in tmux session `name`; output goes to ~/vcc/logs/vcc_2026_4/<name>.log."""
    session = 'vcc4_' + name
    if subprocess.run(['tmux', 'has-session', '-t', session], stderr=subprocess.DEVNULL).returncode == 0:
        raise RuntimeError(f'{session} is already running; wait for completion')
    log, sh = Path(ENV['LOGS']) / f'{name}.log', Path(ENV['LOGS']) / f'{name}.sh'
    exports = ''.join(f'export {k}={shlex.quote(v)}\n' for k, v in ENV.items())
    sh.write_text(f'#!/bin/bash\n{exports}exec > {shlex.quote(str(log))} 2>&1\n'
                  f'trap \'echo "EXIT=$?"\' EXIT\nset -eo pipefail\ncd ~/vcc\n{cmd}\n')
    subprocess.run(['tmux', 'new-session', '-d', '-s', session, 'bash', str(sh)], check=True)
    print(f'started "{name}" in the background. Check with: status("{name}")')

def status(name, n=20):
    session = 'vcc4_' + name
    alive = subprocess.run(['tmux', 'has-session', '-t', session], stderr=subprocess.DEVNULL).returncode == 0
    log = Path(ENV['LOGS']) / f'{name}.log'
    lines = log.read_text().splitlines() if log.exists() else []
    state = 'RUNNING' if alive else ('FINISHED OK' if lines and lines[-1] == 'EXIT=0' else ('NOT STARTED' if not lines else 'STOPPED - read the log below'))
    print(f'== {name}: {state}\n' + '\n'.join(lines[-n:]))

## 1. Raw cells: one file per cell line
`scripts/build_raw_cells.py` streams each source's raw counts, **keeps every cell** (no UMI cut yet: `total_umi` is stored for one later; `--min-umi` applies one at build time), and writes `lib/cells/<NAME>.h5ad` (integer counts, compressed sparse) with the same labels everywhere:

| obs column | meaning |
|---|---|
| `cell_line`, `dataset`, `condition` | the cell line / type, the source study, a study condition where the source has one |
| `target_gene` | the knocked-down gene; `non-targeting` for every control; `A+B` for a pair |
| `n_targets` | 0 control, 1 single knockdown, 2 pair, −1 no assigned guide (`target_gene` = `unassigned`) |
| `perturbation`, `guide`, `batch` | the source's own label, guide and batch, unchanged |
| `total_umi`, `umi_source` | UMIs per cell: `library` = the source's total over all genes (Replogle / Nadig release only ~8,250 genes, so a matrix sum would undercount), `matrix` = sum over the stored genes |
| `n_genes_detected` | genes with ≥ 1 count |

Genes are each source's own (index = symbol, made unique; `gene_symbol`, `gene_id` = Ensembl where available), not yet matched to the challenge's 18,533. PerturBase cells with no assigned guide are kept and labelled `unassigned`.

| File | Cell line | Source | Notes |
|---|---|---|---|
| `K562` | K562 | Replogle 2022, genome-wide | ~8,250 expressed genes released |
| `RPE1` | RPE1 | Replogle 2022, essential | ~8,250 genes |
| `HepG2`, `Jurkat` | HepG2, Jurkat | Nadig 2025 | ~8,250 genes |
| `H1` | H1 (hESC) | Arc 2025 | our validation line |
| `HCT116_targets`, `HEK293T_targets` | HCT116, HEK293T | X-Atlas/Orion 2025 | **subset**: the 300 A–C targets + controls only; the full release is ~520 GB |
| `hESC` | hESC | Genga 2019 | during endoderm differentiation; arrayed |
| `HFF` | foreskin fibroblasts | Hein 2021 | the uninfected 00h cells only (knockdowns and controls; every infected cell removed) |

Dropped after review (most knockdowns lowered their target by less than 50%): the Tian 2019 / 2021 iPSC and neuron screens, MCF10A (Hill 2018) and LUHMES (Lalli 2020). CD4 T cells (Marson) have no raw cells here, only effect tables, so they are not in this set. Builds missing files only; raw downloads it fetched are deleted after each successful build (the bucket, GEO, figshare and PerturBase keep the originals). Genome-wide K562 needs ~66 GB of free disk while it is converted.

In [ ]:
ALL = ['K562', 'H1', 'HCT116_targets', 'HEK293T_targets', 'HepG2', 'Jurkat', 'RPE1',
       'hESC', 'HFF']
manifest = Path(ENV['CELLS']) / 'manifest.csv'
done = set(pd.read_csv(manifest).name) if manifest.exists() else set()
todo = [n for n in ALL if n not in done or not (Path(ENV['CELLS']) / f'{n}.h5ad').exists()]
print('already built:', sorted(done & set(ALL)) or 'none'); print('to build:', todo or 'none')
if todo:
    bg('raw_cells', f'"$PY" scripts/build_raw_cells.py --fetch --delete-raw --datasets {" ".join(todo)}')

In [ ]:
status('raw_cells', n=5)
if manifest.exists():
    m = pd.read_csv(manifest).set_index('name')
    display(m[['cell_line', 'dataset', 'cells_after', 'cells_below_1000_umi', 'unassigned_cells', 'knockdowns', 'pairs',
               'control_cells', 'genes', 'median_umi', 'umi_source', 'gb']].loc[[n for n in ALL if n in m.index]])